# Controlled experiment: recursive `lag_168`

Цель — сравнить текущий tuned LightGBM с тем же набором `SET_FEATURES + CALENDAR_FEATURES + WEATHER_FEATURES` и единственным дополнительным признаком `lag_168(route,t) = boardings(route,t−168h)`. Единица прогноза — `(route, date, hour)`, target — `boardings`; три expanding temporal folds и WAPE взяты из HPO. Нового final fit и submission здесь нет.

## Доступность и leakage

Первые 168 часов train не имеют недельной истории: `NaN` означает неизвестное значение, а `0` означал бы наблюдавшийся нулевой поток. Для route 5 валидной target history нет, поэтому его lag всегда missing, а итоговый прогноз равен `0`.

Первые 7 дней holdout используют actual из pre-origin train; после этого `t−168h` лежит внутри holdout. Там доступны только предыдущие predictions после non-negative clipping, rounding и правила route 5. Добавление actual validation `boardings` в state или `concat(train, validation) → shift(168)` дало бы leakage. Ошибка прогноза может передаваться следующей неделе, поэтому важно отдельно оценить дни 8–28 и 29–61.

In [1]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
while not (root / 'ml' / 'src' / 'model_selection.py').exists():
    if root == root.parent:
        raise FileNotFoundError('Project root not found')
    root = root.parent
sys.path.insert(0, str(root / 'ml' / 'src'))
from recursive_lag_experiment import run_experiment, ARTIFACT

result = run_experiment()
print('Reference reproduced from HPO; result:', ARTIFACT.relative_to(root))
print('Classification:', result['classification'])

Reference reproduced from HPO; result: ml\experiments\results\recursive_lag_168.json
Classification: RECURSIVE_LAG168_REJECTED


## Sanity checks до интерпретации метрик

Доли считаются среди всех строк, включая 10% route 5 с missing lag. В первую неделю 90% lag берётся из pre-origin actual, дальше 90% — из recursive predictions. Сам recursive runner принимает validation без колонки `boardings` и проверяет источники state.

In [2]:
coverage = pd.DataFrame([{'fold': f['fold'], 'train_available': f['train_lag']['available_fraction'],
    **f['validation_lag']['source_fractions'],
    'validation_actual_in_state': f['validation_lag']['validation_actual_in_state'],
    'route_5_lag_nonmissing': f['validation_lag']['route_5_lag_nonmissing']} for f in result['folds']])
display(coverage)
display(pd.DataFrame([{'fold': f['fold'], **week} for f in result['folds']
                      for week in f['validation_lag']['weekly']]))
assert not coverage.validation_actual_in_state.any()
assert coverage.route_5_lag_nonmissing.eq(0).all()
assert all(f['train_lag']['route_5_missing'] for f in result['folds'])

,fold,train_available,actual_pre_origin,recursive_prediction,missing_route_5,validation_actual_in_state,route_5_lag_nonmissing
0,jan_apr_to_may_jun,0.847500,0.103279,0.796721,0.1,False,0
1,jan_jun_to_jul_aug,0.865193,0.101613,0.798387,0.1,False,0
2,jan_aug_to_sep_oct,0.874074,0.103279,0.796721,0.1,False,0


,fold,week,days,rows,actual_pre_origin_fraction,recursive_prediction_fraction,missing_route_5_fraction
0,jan_apr_to_may_jun,1,"[1, 7]",1680,0.9,0.0,0.1
1,jan_apr_to_may_jun,2,"[8, 14]",1680,0.0,0.9,0.1
2,jan_apr_to_may_jun,3,"[15, 21]",1680,0.0,0.9,0.1
3,jan_apr_to_may_jun,4,"[22, 28]",1680,0.0,0.9,0.1
4,jan_apr_to_may_jun,5,"[29, 35]",1680,0.0,0.9,0.1
5,jan_apr_to_may_jun,6,"[36, 42]",1680,0.0,0.9,0.1
6,jan_apr_to_may_jun,7,"[43, 49]",1680,0.0,0.9,0.1
7,jan_apr_to_may_jun,8,"[50, 56]",1680,0.0,0.9,0.1
8,jan_apr_to_may_jun,9,"[57, 61]",1200,0.0,0.9,0.1
9,jan_jun_to_jul_aug,1,"[1, 7]",1680,0.9,0.0,0.1


In [3]:
display(pd.DataFrame([{'fold': f['fold'], 'reference_wape': f['reference']['wape'],
    'recursive_wape': f['candidate']['wape'],
    'delta_wape': f['candidate']['wape'] - f['reference']['wape']} for f in result['folds']]))
display(pd.DataFrame([{'bucket': bucket, 'reference_wape': item['reference']['wape'],
    'recursive_wape': item['candidate']['wape'], 'delta_wape': item['delta_wape']}
    for bucket, item in result['pooled_horizons'].items()]))
display(pd.DataFrame([{'route': route, 'reference_wape': metric['wape'],
    'recursive_wape': result['summaries']['candidate']['per_route'][route]['wape']}
    for route, metric in result['summaries']['reference']['per_route'].items()]))
print('Pooled WAPE:', result['summaries']['reference']['pooled_wape'],
      result['summaries']['candidate']['pooled_wape'])
print('Pooled absolute error:', result['summaries']['reference']['pooled_absolute_error'],
      result['summaries']['candidate']['pooled_absolute_error'])
print('WAPE-score:', result['summaries']['reference']['wape_score'],
      result['summaries']['candidate']['wape_score'])

,fold,reference_wape,recursive_wape,delta_wape
0,jan_apr_to_may_jun,0.141207,0.189901,0.048693
1,jan_jun_to_jul_aug,0.205422,0.198844,-0.006578
2,jan_aug_to_sep_oct,0.114021,0.122807,0.008787


,bucket,reference_wape,recursive_wape,delta_wape
0,days_1_7,0.123722,0.179136,0.055414
1,days_8_28,0.153421,0.169625,0.016203
2,days_29_61,0.154648,0.164822,0.010173


,route,reference_wape,recursive_wape
0,1,0.124881,0.152070
1,5,NaN,NaN
2,7,0.271423,0.279044
3,11,0.126133,0.141426
4,12,0.121498,0.139940
5,17,0.101852,0.115630
6,25,0.193525,0.215761
7,26,0.165674,0.178832
8,28,0.184384,0.213251
9,50,0.212339,0.236652


Pooled WAPE: 0.1509561111066385 0.1682069754284296
Pooled absolute error: 5287697 5891961
WAPE-score: 0.8490438888933615 0.8317930245715703


## Решение

Классификация задаётся по pooled WAPE, трём folds и двум поздним horizon buckets. Первая неделя сама по себе не может подтвердить гипотезу. Точное правило и результаты сохраняются в JSON рядом с остальными experiment results; accepted feature set не меняется.